# Machine Learning Assignment 2 — NPHA Model Training
Leakage-safe training and evaluation of five classifiers using an 80:20 stratified split.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, confusion_matrix, f1_score, matthews_corrcoef, precision_score, recall_score, roc_auc_score)
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler, label_binarize
from sklearn.tree import DecisionTreeClassifier


In [ ]:
RANDOM_STATE = 42
TARGET_COLUMN = 'Number of Doctors Visited'
DATASET_PATH = Path('NPHA-doctor-visits.csv')

if not DATASET_PATH.exists():
    raise FileNotFoundError(f'Place {DATASET_PATH.name} in the notebook directory.')

df = pd.read_csv(DATASET_PATH).dropna(axis=0, how='all').dropna(axis=1, how='all')
if TARGET_COLUMN not in df.columns:
    raise ValueError(f'Missing target column: {TARGET_COLUMN}')

predictor_columns = [c for c in df.columns if c != TARGET_COLUMN]
df[predictor_columns] = df[predictor_columns].replace(-1, np.nan)
X = df.drop(columns=[TARGET_COLUMN])
y = df[TARGET_COLUMN]
print('Shape:', df.shape)
print('Classes:', sorted(y.unique()))


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y
)

test_df = X_test.copy()
test_df[TARGET_COLUMN] = y_test
test_df.to_csv('test_data.csv', index=False)
print('Training:', X_train.shape, 'Test:', X_test.shape)


In [ ]:
numeric_features = X_train.select_dtypes(include=[np.number]).columns.tolist()
categorical_features = X_train.select_dtypes(exclude=[np.number]).columns.tolist()

numeric_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
])
categorical_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False)),
])
transformers = []
if numeric_features:
    transformers.append(('num', numeric_pipeline, numeric_features))
if categorical_features:
    transformers.append(('cat', categorical_pipeline, categorical_features))
preprocessor = ColumnTransformer(transformers=transformers)

models = {
    'Logistic Regression': LogisticRegression(max_iter=2000, random_state=RANDOM_STATE),
    'Decision Tree': DecisionTreeClassifier(random_state=RANDOM_STATE),
    'K-Nearest Neighbors': KNeighborsClassifier(n_neighbors=5),
    'Naive Bayes': GaussianNB(),
    'Random Forest': RandomForestClassifier(n_estimators=200, random_state=RANDOM_STATE, n_jobs=-1),
}


In [ ]:
def multiclass_auc(y_true, probabilities, classes):
    try:
        if len(classes) == 2:
            return roc_auc_score(y_true, probabilities[:, 1], labels=classes)
        y_bin = label_binarize(y_true, classes=classes)
        return roc_auc_score(y_bin, probabilities, average='weighted', multi_class='ovr')
    except ValueError:
        return np.nan

results = []
predictions = {}
fitted_models = {}

for name, estimator in models.items():
    pipeline = Pipeline([('preprocessor', preprocessor), ('classifier', estimator)])
    pipeline.fit(X_train, y_train)
    y_pred = pipeline.predict(X_test)
    y_prob = pipeline.predict_proba(X_test)
    classes = pipeline.named_steps['classifier'].classes_
    results.append({
        'ML Model Name': name,
        'Accuracy': accuracy_score(y_test, y_pred),
        'AUC': multiclass_auc(y_test, y_prob, classes),
        'Precision': precision_score(y_test, y_pred, average='weighted', zero_division=0),
        'Recall': recall_score(y_test, y_pred, average='weighted', zero_division=0),
        'F1 Score': f1_score(y_test, y_pred, average='weighted', zero_division=0),
        'MCC': matthews_corrcoef(y_test, y_pred),
    })
    predictions[name] = y_pred
    fitted_models[name] = pipeline

results_df = pd.DataFrame(results)
results_df.to_csv('model_comparison_results.csv', index=False)
results_df.round(4)


In [ ]:
selected_model_name = 'Logistic Regression'
labels = fitted_models[selected_model_name].named_steps['classifier'].classes_
cm = confusion_matrix(y_test, predictions[selected_model_name], labels=labels)

plt.figure(figsize=(6, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=labels, yticklabels=labels)
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title(f'Confusion Matrix - {selected_model_name}')
plt.tight_layout()
plt.show()
